# Task 2

# Cканирование хромосомы человека
Цель: научиться работать с реальными геномными данными и модулем Bio.motifs.
Задачи:
1. Используя модуль Bio.motifs из Biopython, создайте мотив из последовательностей Задания 1.
2. Загрузите файл chr1.fa (первая хромосома человека) с помощью SeqIO.parse.
3. Извлеките первые 1 000 000 нуклеотидов для анализа.
4. Используйте метод m.pwm.search(sequence, threshold=...) для поиска сайтов связывания.
5. Найдите все хиты (позицию и скор) со скором больше 5.0.
6. Повторите поиск для обратной комплементарной цепи.
Подсказка: Транскрипционные факторы могут связываться на обеих цепях ДНК!
Ответ: Код поиска по хромосоме. Список найденных хитов (позиция, цепь, скор), превышающих порог
5.0, сохраненный в текстовый файл или выведенный в ноутбуке.

In [1]:
from Bio import SeqIO, motifs
import numpy as np

In [14]:
sequences = [ "GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC", "CAGGTTGGC"]
m = motifs.create(sequences)
pwm = m.pwm

In [15]:
chr_data = list(SeqIO.parse('/Users/veronikaaksinina/Documents/bioinf_sem_26/chr21.fasta', 'fasta'))
chromosome_seq = chr_data[0].seq

In [16]:
chr_part = chromosome_seq[:1_000_000]

In [17]:
# ===== 4. ФУНКЦИЯ ПОИСКА ЧЕРЕЗ PWM =====
def search_hits_pwm(sequence, pwm, motif_length, threshold, strand='+'):
    """
    Ищет позиции, используя PWM (словарь словарей)
    """
    hits = []
    
    for i in range(len(sequence) - motif_length + 1):
        subseq = sequence[i:i+motif_length]
        score = 0.0
        valid = True
        
        for j, nucleotide in enumerate(subseq):
            # Проверяем, что нуклеотид есть в PWM
            if nucleotide not in pwm[j]:
                valid = False
                break
            score += pwm[j][nucleotide]
        
        if valid and score >= threshold:
            hits.append((i, strand, round(float(score), 2)))
    
    return hits

In [18]:
motif_len = m.length
threshold = 3.0  # можешь менять

In [19]:
print("\n🔍 Ищем на прямой цепи...")
hits_fwd = search_hits_pwm(chr_part, pwm, motif_len, threshold, strand='+')
print(f"✅ Найдено {len(hits_fwd)} хитов на прямой цепи")


🔍 Ищем на прямой цепи...
✅ Найдено 0 хитов на прямой цепи


In [20]:
print("🔍 Ищем на обратной цепи...")
rev_comp = chr_part.reverse_complement()
hits_rev = search_hits_pwm(rev_comp, pwm, motif_len, threshold, strand='-')
print(f"✅ Найдено {len(hits_rev)} хитов на обратной цепи")

🔍 Ищем на обратной цепи...
✅ Найдено 0 хитов на обратной цепи


In [21]:
all_hits = hits_fwd + hits_rev
all_hits.sort(key=lambda x: x[2], reverse=True)

print(f"\n📊 ВСЕГО НАЙДЕНО ХИТОВ: {len(all_hits)}")


📊 ВСЕГО НАЙДЕНО ХИТОВ: 0


In [22]:
with open('hits_chr21.txt', 'w') as f:
    f.write("Позиция\tЦепь\tСкор\n")
    for pos, strand, score in all_hits:
        f.write(f"{pos}\t{strand}\t{score}\n")

print("✅ Результаты сохранены в 'hits_chr21.txt'")

✅ Результаты сохранены в 'hits_chr21.txt'


In [23]:
print("\n🏆 ТОП-10 ХИТОВ (по скору):")
for i, (pos, strand, score) in enumerate(all_hits[:10], 1):
    print(f"{i:2}. Позиция: {pos:8} | Цепь: {strand} | Скор: {score:6.2f}")


🏆 ТОП-10 ХИТОВ (по скору):


In [24]:
# ===== 11. ЕСЛИ НЕТ ХИТОВ =====
if len(all_hits) == 0:
    print("\n⚠️ Хитов не найдено! Попробуй:")
    print("   - Уменьшить порог (например, до 3.0 или 1.0)")
    print("   - Проверить, что в последовательностях мотива нет ошибок")
    print("   - Убедиться, что в хромосоме есть нуклеотиды A/C/G/T")


⚠️ Хитов не найдено! Попробуй:
   - Уменьшить порог (например, до 3.0 или 1.0)
   - Проверить, что в последовательностях мотива нет ошибок
   - Убедиться, что в хромосоме есть нуклеотиды A/C/G/T
